# 09 - Cierre del EDA y priorización de KPIs

- Cierra la etapa descriptiva y reúne las conclusiones de PE2.
- Vuelve explícitas las unidades, denominadores, cobertura y limitaciones de cada KPI.
- Prioriza indicadores por las tres decisiones del Instituto de Vivienda.
- Identifica comunas para diagnóstico adicional con reglas transparentes; no es un índice normativo.
- Insumos: base enriquecida, KPIs por barrio y comuna, síntesis de hipótesis de 08.
- Salidas: tablas de jerarquización y señales exploratorias en `data/processed/reports/`.

## 1. Carga y controles

- Se reutilizan exclusivamente las salidas documentadas de 07 y 08.
- Se comprueba la clave publicación-operación y la cobertura del universo territorial.
- Las cantidades de avisos no se interpretan como proporciones del stock.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src import config

REPORTS = config.REPORTS_DIR
base = pd.read_csv(
    config.PROCESSED_DIR / "df_publicaciones_enriquecidas_v1.csv",
    low_memory=False,
    dtype={"id_publicacion": str}
)
kb = pd.read_csv(config.PROCESSED_DIR / "kpis_barrio_v1.csv")
kc = pd.read_csv(config.PROCESSED_DIR / "kpis_comuna_v1.csv")
sintesis_08 = pd.read_csv(REPORTS / "eda08_sintesis_hipotesis_v1.csv")

assert not base.duplicated(["id_publicacion", "tipo_operacion_norm"]).any()
assert kb["barrio_oficial"].nunique() == len(kb)
assert kc["comuna"].nunique() == len(kc)

print("Base enriquecida:", base.shape)
print("Barrios con KPI:", kb["barrio_oficial"].nunique())
print("Comunas con KPI:", kc["comuna"].nunique())
display(sintesis_08)

## 2. De hipótesis a evidencia utilizable

- Se conserva el estado de cada hipótesis del notebook 08.
- Se separa señal observada de evidencia insuficiente.
- En H2a, el tamaño medio del hogar es un proxy indirecto; no mide hogares unipersonales.
- H4 no permite concluir desplazamiento ni sustitución con datos transversales.

In [ ]:
display(sintesis_08[["hipotesis", "unidad", "evidencia", "lectura", "limitaciones"]])
sintesis_08.to_csv(REPORTS / "pe3_sintesis_hipotesis_v1.csv", index=False)

## 3. Jerarquía de KPIs

- Criterios: vínculo con una decisión, denominador interpretable, cobertura y limitaciones.
- La tabla es una priorización metodológica; no asigna pesos ni agrega dimensiones en un índice único.
- La participación de publicaciones describe la muestra recolectada, no el mercado completo.

In [ ]:
catalogo_kpis = pd.DataFrame([
    ["Disponibilidad residencial publicada", "alquiler_por_1000_viviendas", "Comuna", "Fortalecer alquiler permanente", "Conteo de avisos / viviendas Censo 2022; no es stock disponible"],
    ["Concentración temporaria", "airbnb_por_1000_viviendas; airbnb_por_km2", "Comuna / barrio", "Monitorear alquiler temporario", "Airbnb es una fuente específica y un corte de 2026"],
    ["Presión temporaria observada", "pct_viviendas_uso_temporal; brecha Airbnb/permanente", "Comuna / barrio", "Monitorear y producir información", "Censo 2022 y publicaciones tienen fechas y universos distintos"],
    ["Adecuación tipológica", "participación 1-2 ambientes; tamano_medio_hogar", "Comuna", "Priorizar intervención habitacional", "Tamaño medio del hogar es un proxy, no composición individual"],
    ["Accesibilidad urbana", "dist_transporte_masivo_m; densidades de servicios", "Publicación / barrio", "Priorizar infraestructura", "Distancias lineales y cobertura desigual por fuente"],
    ["Asequibilidad/precio publicado", "mediana_precio_m2_venta_usd; mediana_precio_m2_alquiler_usd", "Barrio", "Orientar diagnóstico habitacional", "Precios pedidos; no son precios de cierre ni ingresos de hogares"],
], columns=["dimension", "kpi", "unidad", "decision_informada", "limitacion_clave"])
display(catalogo_kpis)
catalogo_kpis.to_csv(REPORTS / "pe3_jerarquia_kpis_v1.csv", index=False)

## 4. Señal exploratoria para diagnóstico comunal

- Se consideran conjuntamente concentración de Airbnb, baja oferta permanente publicada relativa a viviendas y alquiler mediano elevado.
- Los umbrales son cuartiles internos del corte disponible, no metas de política.
- La intersección identifica casos para investigar; no acredita déficit, desplazamiento ni necesidad de una intervención específica.
- Se informa el número de comunas y se muestran los valores continuos para evitar que el semáforo oculte magnitudes.

In [ ]:
alquiler_comuna = (
    base[(base["tipo_operacion_norm"] == "alquiler")
         & base["precio_m2_usd"].notna()
         & ~base["precio_m2_usd_fuera_rango_flag"]]
    .groupby("comuna_oficial")["precio_m2_usd"].median()
)
alquiler_comuna.index = pd.to_numeric(alquiler_comuna.index, errors="coerce")
diag = kc.set_index("comuna").copy()
diag["mediana_alquiler_usd_m2_mes"] = alquiler_comuna
diag = diag.dropna(subset=[
    "airbnb_por_1000_viviendas",
    "alquiler_por_1000_viviendas",
    "mediana_alquiler_usd_m2_mes"
])
diag["airbnb_alto"] = diag["airbnb_por_1000_viviendas"] >= diag["airbnb_por_1000_viviendas"].quantile(.75)
diag["alquiler_publicado_bajo"] = diag["alquiler_por_1000_viviendas"] <= diag["alquiler_por_1000_viviendas"].quantile(.25)
diag["alquiler_caro"] = diag["mediana_alquiler_usd_m2_mes"] >= diag["mediana_alquiler_usd_m2_mes"].quantile(.75)
diag["senales_presentes"] = diag[["airbnb_alto", "alquiler_publicado_bajo", "alquiler_caro"]].sum(axis=1)
diag["candidata_diagnostico_adicional"] = diag["senales_presentes"] == 3

print("Comunas evaluables:", len(diag), "de", len(kc))
print("Comunas con las tres señales:", int(diag["candidata_diagnostico_adicional"].sum()))
display(diag[[
    "airbnb_por_1000_viviendas", "alquiler_por_1000_viviendas",
    "mediana_alquiler_usd_m2_mes", "senales_presentes",
    "candidata_diagnostico_adicional"
]].sort_values(["senales_presentes", "airbnb_por_1000_viviendas"], ascending=[False, False]).round(2))

fig, ax = plt.subplots(figsize=(7.2, 4.6))
for comuna, fila in diag.iterrows():
    color = "#c45a46" if fila["candidata_diagnostico_adicional"] else "#4b78a8"
    ax.scatter(fila["airbnb_por_1000_viviendas"], fila["alquiler_por_1000_viviendas"],
               color=color, s=48, edgecolor="white", linewidth=.7)
    ax.annotate("C" + str(int(comuna)),
                (fila["airbnb_por_1000_viviendas"], fila["alquiler_por_1000_viviendas"]),
                xytext=(4, 3), textcoords="offset points", fontsize=7)
ax.axvline(diag["airbnb_por_1000_viviendas"].quantile(.75), color="#888888", linestyle="--", linewidth=1)
ax.axhline(diag["alquiler_por_1000_viviendas"].quantile(.25), color="#888888", linestyle="--", linewidth=1)
ax.set_xlabel("Avisos Airbnb por 1.000 viviendas")
ax.set_ylabel("Avisos de alquiler permanente por 1.000 viviendas")
ax.set_title("Comunas con señales para diagnóstico adicional (n = {})".format(len(diag)))
ax.text(.01, -.22, "Cortes internos: Airbnb ≥ P75; alquiler permanente ≤ P25. No son umbrales normativos.",
        transform=ax.transAxes, fontsize=8)
plt.tight_layout()
plt.show()
diag.reset_index().to_csv(REPORTS / "pe3_senales_priorizacion_comunas_v1.csv", index=False)

## 5. Lectura para las decisiones

- **Alquiler permanente:** contrastar baja intensidad de avisos con hogares inquilinos, stock censal y tamaño de muestra.
- **Alquiler temporario:** revisar fuentes por separado; Airbnb y Mercado Libre no representan el mismo producto.
- **Vivienda o accesibilidad:** cruzar la señal habitacional con tipología, transporte y servicios; no trasladar un promedio comunal a cada hogar.
- **Fiscalización o regulación:** los datos actuales no identifican cumplimiento registral ni causalidad; primero se requiere información administrativa y temporal.
- **Próximo paso:** usar el perfil multidimensional por barrio para explorar si hay grupos territoriales interpretables.

## 6. Alcance macroeconómico

- Los precios publicados en ARS se convierten a USD con MEP por fecha del aviso.
- Esa operación homogeneiza moneda nominal en el corte; no elimina inflación ni cambios en el valor real del dólar.
- El relevamiento de agosto-septiembre de 2026 no permite estimar cambios temporales ni efectos de reformas.
- Se requiere una serie de snapshots y series de inflación y tipo de cambio alineadas para evaluar esas preguntas.